# 2. Build and run a query

Queries are built from one or more **clauses** (`picsure::buildClause()`) which can stand alone or be combined into a group with `picsure::buildClauseGroup()`. `picsure::runQuery()` executes them.

This notebook builds a single categorical FILTER from a dictionary search result and asks the backend for a participant count.

In [ ]:
library(picsure)

In [ ]:
token_file <- "token.txt"
my_token <- readLines(token_file, warn = FALSE)[1]

In [ ]:
authorized_session <- picsure::connect(
  platform = picsure::Platform$BDC_AUTHORIZED,
  token    = my_token
)

## Find a variable inside a single study

In [ ]:
framingham_facet <- picsure::facets(authorized_session)
picsure::addFacet(framingham_facet, "dataset_id", "tutorial-biolincc_framingham")

In [ ]:
tutorial_search_results <- picsure::searchDictionary(
  authorized_session,
  "Current cigarette smoking at exam",
  facets = framingham_facet
)
tutorial_search_results$values

In [ ]:
cursmoke_var <- tutorial_search_results[tutorial_search_results$name == "CURSMOKE", ]
cursmoke_var

In [ ]:
raw_values <- cursmoke_var$values[[1]]
raw_values

## Build a categorical FILTER

Pass the concept path and the categorical values to match. `categories` accepts a vector or list.

In [ ]:
cursmoke <- picsure::buildClause(
  cursmoke_var$conceptPath[[1]],
  type       = picsure::PhenotypicFilterType$FILTER,
  categories = raw_values
)

## Run for a count

Assemble the clause into a `Query` with `picsure::buildQuery()`, then run it (`runQuery()` also accepts a bare clause or group directly). `type = picsure::QueryType$COUNT` returns a `CountResult` with `$value`, `$margin`, and `$cap` slots; `$value` is `NULL` for small-cohort obfuscation.

In [ ]:
full_query <- picsure::buildQuery(phenotypicFilter = cursmoke)

count <- picsure::runQuery(authorized_session, full_query, type = picsure::QueryType$COUNT)
count$value

## Select output columns with `includeConcepts`

`includeConcepts` adds concepts as output columns without changing which participants match — the replacement for the old `SELECT` clause. Here we pull participant-level rows for the cohort, returning the smoking concept as a column.

In [ ]:
participant_query <- picsure::buildQuery(
  phenotypicFilter = cursmoke,
  includeConcepts  = cursmoke_var$conceptPath[[1]]
)

participants <- picsure::runQuery(
  authorized_session, participant_query, type = picsure::QueryType$PARTICIPANT
)
head(participants)